<table>
  <tr>
    <td><div align="left"><font size="30">Planar homography</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200" height="158" style="height:auto"></td>
  </tr>
</table>

(c) Peter Corke 2024

Robotics, Vision & Control: Python, see Section 13.6.2

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`). If you're running locally you need to have these packages installed.

<div class="alert alert-info">
If you're running on CoLab or in the browser we have to first install machinevisiontoolbox, which is not preinstalled, so the next cell will be slow. In the browser environment it is not possible to examine pixel values by hovering the mouse cursor over an image (this is a limitation of Matplotlib in a Pyodide Web Worker).
</div>

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
%matplotlib ipympl

import numpy as np
np.set_printoptions(linewidth=120, formatter={'float': lambda x: f"{x:8.4g}" if abs(x) > 1e-10 else f"{0:8.4g}"})

from spatialmath import SE3
from spatialmath.base import e2h, h2e, homtrans, plot_sphere
from machinevisiontoolbox import CentralCamera


***

We define a central perspective camera (see camera.ipynb for more details on this), that is position up high, looking obliquely downward at the ground

In [ ]:
camera = CentralCamera(f=0.012, rho=10e-6, imagesize=1000, 
        pose=SE3(0, 0, 8) * SE3.Rx(-2.8))

And we can plot the camera in the 3D world

In [ ]:
ax = camera.plot(scale=2, shape='camera', color='k', frame=True)
ax.set_xlim(-8, 12)
ax.set_ylim(-10, 10)
ax.set_zlim(0, 10)

A shape on the ground plane is defined by a set of 2D coordinates

In [ ]:
P = np.column_stack([[-1, 1], [-1, 2], [ 2,2], [2, 1]])
P

To obtain the coordinates of the points in 3D, we augment each column with a zero, since the ground plane is defined by $z=0$

In [ ]:
P0 = np.vstack([P, np.zeros((4,))])
P0

Now we can project the 3D ground plane points onto the image plane

In [ ]:
camera.project_point(P0)

The homography is computed from the camera matrix by deleting column two (the z column)

In [ ]:
H = np.delete(camera.C(), 2, axis=1)
H

We can use this matrix to directly compute the image plane points, by transforming the homogeneous ground plane points

In [ ]:
h2e(H @ e2h(P))

or more simply

In [ ]:
homtrans(H, P)

which first converts `P` to homogeneous form, performs the multiplication, then converts the resulting homogeneous coordinates to Euclidean.


H is square and of full rank, so it is invertible. This means that we can perform the inverse mapping, from the image plane
to the ground plane.

The camera has a 1000 x 1000 image plane so the coordinates of its corners are


In [ ]:
p = np.column_stack([[0, 0], [0, 1000], [1000, 1000], [1000, 0]])
p

and on the ground plane these are the points

In [ ]:
Pi = homtrans(np.linalg.inv(H), p)
Pi

Now we can overlay the corners of the camera's field of view onto the "world view" of the imaging setup that we showed earlier

In [ ]:
ax = camera.plot(scale=2, shape='camera', color='k', frame=True)
k = [0, 1, 2, 3, 0]
ax.plot(Pi[0, k], Pi[1, k], np.zeros(5), 'b--')


Scroll back up to the previous figure to see the blue dashed line representing the field of view.